# ViRead VietOCR V2 — small training smoke run

This is a **local diagnostic only** for trying a few already verified Flow 2 samples while the official Locked Test is still collecting independent pages. It reads the Admin dataset and active checkpoint, trains on a small Train/Validation subset, and stores the output in a separate Google Drive folder.

It does **not** create a training job/run, upload a candidate, run a Locked Test benchmark, or change the active model. A successful run only confirms that this small subset passed download, preprocessing, dataloader, GPU training, validation, and local checkpoint writing. Keep the production Locked Test requirement unchanged.

Before running: choose a Colab GPU runtime, add Colab Secrets `VIREAD_API_BASE` and `VIREAD_ADMIN_BEARER`, then select **Runtime → Run all**. The Admin dataset endpoint may materialize an immutable snapshot of existing verified samples if one does not exist; it does not change source OCR pages. This notebook creates no job or run. If the production Locked Test is already READY, the separate scheduler may independently enqueue its normal full training job from the complete registered dataset.


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import hashlib, json, os, subprocess, sys, time, requests

def get_secret(name):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if not value:
        raise RuntimeError(f"Missing Colab Secret: {name}")
    return value

API_BASE = get_secret("VIREAD_API_BASE").rstrip("/")
ADMIN_BEARER = get_secret("VIREAD_ADMIN_BEARER")
REPOSITORY_URL = "https://github.com/trancuong3/capstone1.git"
REPO = Path("/content/viread-capstone1-smoke")
WORK = Path("/content/drive/MyDrive/ViRead/VietOCR-V2-smoke")
drive.mount("/content/drive")
WORK.mkdir(parents=True, exist_ok=True)

if not (REPO / "ai/vietocr_v2/train.py").is_file():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY_URL, str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", str(REPO / "ai/vietocr_v2/requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-deps", "vietocr==0.3.12"], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

from ai.vietocr_v2.contracts import DatasetRow, write_jsonl, sha256_file
from ai.vietocr_v2.artifact_store import assemble_shards
from ai.vietocr_v2.train import prepare_vietocr_training_runtime, run_training

runtime = prepare_vietocr_training_runtime()
print(f"Dependencies ready: NumPy {runtime['numpy_version']}, Pillow {runtime['pillow_version']}, PyTorch {runtime['torch_version']}, CUDA available={runtime['cuda_available']}")
if not runtime["cuda_available"]:
    raise RuntimeError("No GPU is attached. In Colab choose Runtime → Change runtime type → T4 GPU, reconnect, then run all cells again.")

HEADERS = {"Authorization": f"Bearer {ADMIN_BEARER}"}
def admin_call(method, path, **kwargs):
    response = requests.request(method, API_BASE + path, headers=HEADERS, timeout=300, **kwargs)
    response.raise_for_status()
    return response.json() if response.content else {}

def download_checked(url, destination, expected_sha256):
    destination = Path(destination)
    if destination.is_file():
        if sha256_file(destination) == expected_sha256:
            return
        raise ValueError("A cached file failed its SHA-256 check; choose a fresh smoke folder.")
    destination.parent.mkdir(parents=True, exist_ok=True)
    digest = hashlib.sha256()
    temporary = destination.with_suffix(destination.suffix + ".download")
    try:
        with requests.get(url, stream=True, timeout=300) as response:
            response.raise_for_status()
            with temporary.open("xb") as output:
                for block in response.iter_content(1024 * 1024):
                    if block:
                        output.write(block)
                        digest.update(block)
        if digest.hexdigest() != expected_sha256:
            raise ValueError("Downloaded file failed its SHA-256 check.")
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)

print("Colab is connected. Secret values are not displayed.")


## 1. Select a small, page-separated subset

The notebook chooses up to 3 page groups and 4 line images per group from each existing Train and Validation split. It never selects holdout or Locked Test rows. A local receipt is built for this subset so the normal trainer can verify exactly what it reads; this receipt is not registered with ViRead.


In [ ]:
dataset_page = admin_call(
    "GET", "/books/admin/viread-training/datasets/flow2-admin-verified",
    params={"limit": 1, "offset": 0},
)
if dataset_page.get("source_flow") != "FLOW1_VERIFIED":
    raise ValueError("The selected dataset is not the Admin-verified Flow 2 snapshot.")
DATASET_VERSION_ID = str(dataset_page["dataset_version_id"])
DATASET_SHA256 = dataset_page["manifest_sha256"]

all_items = []
offset = 0
while True:
    page = admin_call(
        "GET", f"/books/admin/viread-training/datasets/{DATASET_VERSION_ID}/samples",
        params={"limit": 200, "offset": offset},
    )
    if page["dataset"]["version_sha256"] != DATASET_SHA256:
        raise ValueError("The verified training dataset changed during pagination; restart this cell.")
    if page["dataset"]["is_locked_test"]:
        raise ValueError("Refusing to use a Locked Test dataset for a smoke run.")
    all_items.extend(page.get("items", []))
    if page.get("next_offset") is None:
        break
    offset = int(page["next_offset"])

eligible = [item for item in all_items if item.get("split") in {"train", "validation"}]
selected = []
MAX_GROUPS_PER_SPLIT = 3
MAX_SAMPLES_PER_GROUP = 4
for split in ("train", "validation"):
    by_group = {}
    for item in eligible:
        if item["split"] == split:
            by_group.setdefault(str(item["group_id"]), []).append(item)
    chosen_groups = sorted(by_group)[:MAX_GROUPS_PER_SPLIT]
    selected_for_split = []
    for group_id in chosen_groups:
        group_items = sorted(by_group[group_id], key=lambda item: str(item.get("sample_id") or item.get("source_annotation_id") or ""))
        selected_for_split.extend(group_items[:MAX_SAMPLES_PER_GROUP])
    selected.extend(selected_for_split)

split_group_counts = {
    split: len({str(item["group_id"]) for item in selected if item["split"] == split})
    for split in ("train", "validation")
}
split_sample_counts = {
    split: sum(item["split"] == split for item in selected)
    for split in ("train", "validation")
}
if any(split_group_counts[split] < 2 or split_sample_counts[split] < 2 for split in ("train", "validation")):
    raise RuntimeError(f"Need at least two independent page groups in both Train and Validation; found {split_group_counts} groups and {split_sample_counts} samples.")

rows = []
for index, item in enumerate(selected, start=1):
    flow = item.get("source_flow") or page["dataset"]["source_flow"]
    if flow != "FLOW1_VERIFIED":
        raise ValueError("Smoke run requires Admin-verified Flow 2 labels.")
    image_url = item.get("image_url") or item.get("image")
    image_sha = str(item["image_sha256"]).lower()
    image_path = WORK / "images" / image_sha
    download_checked(image_url, image_path, image_sha)
    rows.append(DatasetRow(
        image=str(image_path), label=item["label"], group_id=str(item["group_id"]),
        source_flow=flow, split=item["split"], image_sha256=image_sha,
        verified=True, annotation_id=str(item.get("source_annotation_id") or item.get("sample_id") or index),
    ))

run_stamp = time.strftime("%Y%m%d-%H%M%S", time.gmtime())
RUN_DIR = WORK / f"run-{run_stamp}"
RUN_DIR.mkdir(parents=True, exist_ok=False)
TRAINING_MANIFEST = RUN_DIR / "smoke-training.jsonl"
write_jsonl(TRAINING_MANIFEST, rows)
LOCAL_RECEIPT = RUN_DIR / "smoke-receipt.json"
receipt = {
    "schema": "viread-dataset-v1",
    "source_flow": "FLOW1_VERIFIED",
    "split_seed": "local-smoke-subset-v1",
    "samples": [
        {"source_flow": row.source_flow, "group_id": row.group_id, "split": row.split,
         "image_sha256": row.image_sha256, "label": row.label}
        for row in rows
    ],
}
LOCAL_RECEIPT.write_text(json.dumps(receipt, ensure_ascii=False, sort_keys=True, separators=(",", ":")), encoding="utf-8")
SMOKE_DATA_SHA256 = sha256_file(LOCAL_RECEIPT)
print(f"Subset ready: Train {split_sample_counts['train']} samples / {split_group_counts['train']} pages; Validation {split_sample_counts['validation']} samples / {split_group_counts['validation']} pages.")
print("Only Train and Validation were downloaded. Locked Test was not read.")


## 2. Load the active parent checkpoint

This is a read-only fetch. The smoke run starts from the same active checkpoint the official trainer would use, but its output remains local to the separate smoke folder.


In [ ]:
from ai.vietocr_v2.contracts import ORIGINAL_CHECKPOINT_SHA256

active = admin_call("GET", "/books/admin/viread-training/active-model/artifact")
PARENT = RUN_DIR / f"parent-{active['checkpoint_sha256']}.pth"
if active["profile"] == "ORIGINAL":
    download_checked(active["source_url"], PARENT, active["checkpoint_sha256"])
else:
    shard_dir = RUN_DIR / "parent-shards"
    shard_dir.mkdir(parents=True, exist_ok=True)
    parts = []
    for part in active["chunks"]:
        local = shard_dir / f"part-{int(part['index']):04d}.bin"
        download_checked(part["url"], local, part["sha256"])
        parts.append({"index": int(part["index"]), "name": local.name,
                      "bytes": int(part["bytes"]), "sha256": part["sha256"]})
    assemble_shards({
        "schema": "viread-checkpoint-shards-v1", "filename": PARENT.name,
        "bytes": sum(part["bytes"] for part in parts),
        "sha256": active["checkpoint_sha256"], "chunks": parts,
    }, shard_dir, PARENT)
if sha256_file(PARENT) != active["checkpoint_sha256"]:
    raise ValueError("Active parent checkpoint failed its SHA-256 check.")
print(f"Active parent verified: {active['profile']} / {active['checkpoint_sha256'][:12]}…")


## 3. Run a short local training check

The run uses 20 steps, batch size 4, and one initial VGG freeze pass. Validation runs at the end. The output checkpoint stays under `MyDrive/ViRead/VietOCR-V2-smoke`; no candidate or run is sent back to ViRead.


In [ ]:
from tqdm.auto import tqdm
progress = tqdm(total=20, desc="Local smoke training", unit="step")
def show_progress(step, total):
    progress.update(max(0, step - progress.n))

started = time.time()
try:
    smoke_result = run_training(
        profile="FLOW2_ADMIN_VERIFIED",
        manifest_path=str(TRAINING_MANIFEST),
        output_dir=str(RUN_DIR),
        parent_checkpoint=str(PARENT),
        total_steps=20,
        batch_size=4,
        freeze_epochs=1,
        dataset_receipt_path=str(LOCAL_RECEIPT),
        dataset_version_sha256=SMOKE_DATA_SHA256,
        seed=20261114,
        resume=False,
        use_fp16=True,
        progress_callback=show_progress,
        progress_interval_steps=1,
    )
finally:
    progress.close()
elapsed_seconds = round(time.time() - started, 1)
summary = {
    "mode": "LOCAL_SMOKE_ONLY_NOT_REGISTERED",
    "source_dataset_version_sha256": DATASET_SHA256,
    "smoke_subset_sha256": SMOKE_DATA_SHA256,
    "train_samples": split_sample_counts["train"],
    "train_page_groups": split_group_counts["train"],
    "validation_samples": split_sample_counts["validation"],
    "validation_page_groups": split_group_counts["validation"],
    "steps_completed": smoke_result["steps"],
    "device": smoke_result["device"],
    "validation_loss": smoke_result["best_validation_loss"],
    "active_parent_sha256": active["checkpoint_sha256"],
    "local_checkpoint_sha256": smoke_result["checkpoint_sha256"],
    "elapsed_seconds": elapsed_seconds,
    "result": "TRAINING_PIPELINE_SMOKE_PASSED",
}
(RUN_DIR / "smoke-summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Smoke run complete. This local checkpoint is not registered, benchmarked, or eligible to replace the active model.")
